In [74]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

In [75]:
data = pd.read_csv(
    "/Users/gabrielaslomiany/GIT/Alzheimer Detection/data/data.csv",
    sep=None,
    engine="python"
)
print(data.shape)
data.head()

(1737, 361)


,﻿RID,Test_data,Age,Gender,Year_education,Ethnicity,Race,Marital_status,High_risk_ApoE4,Ventricles,...,Cortical Thickness Standard Deviation of RightLingual,Volume (Cortical Parcellation) of RightMedialOrbitofrontal,Surface Area of RightMedialOrbitofrontal,Cortical Thickness Average of RightMedialOrbitofrontal,Cortical Thickness Standard Deviation of RightMedialOrbitofrontal,Volume (Cortical Parcellation) of RightMiddleTemporal,Surface Area of RightMiddleTemporal,Cortical Thickness Average of RightMiddleTemporal,Cortical Thickness Standard Deviation of RightMiddleTemporal,Volume (WM Parcellation) of FourthVentricle
0,2,0,"74,3",Male,16,Not Hisp/Latino,White,Married,0.0,118233.0,...,"0,694",3835,1622,"2,077","0,746",15683,4272,"3,028","0,649",4396
1,3,0,"81,3",Male,18,Not Hisp/Latino,White,Married,1.0,84599.0,...,"0,591",3681,1734,"1,942","0,696",10387,3316,"2,545","0,686",3304
2,4,1,"67,5",Male,10,Hisp/Latino,White,Married,0.0,39605.0,...,"0,588",4060,1728,"2,18","0,607",11156,3598,"2,67","0,631",1338
3,5,0,"73,7",Male,16,Not Hisp/Latino,White,Married,0.0,34062.0,...,"0,628",5180,1868,"2,543","0,709",11579,3387,"2,911","0,66",1623
4,6,0,"80,4",Female,13,Not Hisp/Latino,White,Married,0.0,39826.0,...,"0,631",3078,1241,"2,141","0,701",9641,2781,"2,9","0,727",1035


In [76]:
cognitive = pd.read_csv(
    "/Users/gabrielaslomiany/GIT/Alzheimer Detection/data/cognitive_score.csv",
    sep=None,
    engine="python"
)
print(cognitive.shape)
cognitive.head()

(1737, 10)


,﻿RID,Test_data,CDRSB,ADAS11,ADAS13,MMSE,RAVLT_immediate,RAVLT_learning,RAVLT_forgetting,RAVLT_perc_forgetting
0,2,0,0,"10,67","18,67",28,44.0,4.0,6.0,"54,5455"
1,3,0,"4,5",22,31,20,22.0,1.0,4.0,100
2,4,1,1,"14,33","21,33",27,37.0,7.0,4.0,"36,3636"
3,5,0,0,"8,67","14,67",29,37.0,4.0,4.0,"44,4444"
4,6,0,"0,5","18,67","25,67",25,30.0,1.0,5.0,"83,3333"


In [77]:
diagnosis = pd.read_csv(
    "/Users/gabrielaslomiany/GIT/Alzheimer Detection/data/diagnosis_target.csv",
    sep=None,
    engine="python"
)
print(diagnosis.shape)
diagnosis.head()

(1737, 4)


,﻿RID,Test_data,Diagnosis,FDG_PET
0,2,0,CN,"1,36926"
1,3,0,AD,"1,09079"
2,4,1,LMCI,NaN
3,5,0,CN,"1,29799"
4,6,0,LMCI,NaN


### Missing data

##### Data datframe

In [78]:
data.isna().sum()

﻿RID                                                            0
Test_data                                                       0
Age                                                             0
Gender                                                          0
Year_education                                                  0
                                                               ..
Volume (Cortical Parcellation) of RightMiddleTemporal           0
Surface Area of RightMiddleTemporal                             0
Cortical Thickness Average of RightMiddleTemporal               0
Cortical Thickness Standard Deviation of RightMiddleTemporal    0
Volume (WM Parcellation) of FourthVentricle                     0
Length: 361, dtype: int64

As data dataframe has 361 columns, checking NaN's is much simpler with function below. Data's share is (1737, 361), and count highest count of missing values is 291, which is about 17%.

In [79]:
missing_values = data.isna().sum()
missing_values = missing_values[missing_values > 0]

missing_values

High_risk_ApoE4                                                  12
Ventricles                                                       82
Hippocampus                                                     248
WholeBrain                                                       48
Entorhinal                                                      271
Fusiform                                                        271
MidTemp                                                         271
Intra cranial volume                                             15
Volume (WM Parcellation) of RightVessel                           2
Volume (Cortical Parcellation) of LeftEntorhinal                  1
Surface Area of LeftEntorhinal                                    1
Cortical Thickness Average of LeftEntorhinal                      1
Cortical Thickness Standard Deviation of LeftEntorhinal           1
Volume (Cortical Parcellation) of LeftParahippocampal             1
Surface Area of LeftParahippocampal             

In [80]:
data[['Hippocampus','Entorhinal', 'Fusiform', 'MidTemp', 'Volume (WM Parcellation) of FifthVentricle']].dtypes


Hippocampus                                   float64
Entorhinal                                    float64
Fusiform                                      float64
MidTemp                                       float64
Volume (WM Parcellation) of FifthVentricle        str
dtype: object

Hippocampus, Entorhinal, Fusiform, MidTemp --> replace with mean, as their dtype is float
Volume (WM Parcellation) of FifthVentricle --> drop

In [81]:
columns = ["Entorhinal", "Fusiform", "MidTemp"]

data[columns] = data[columns].fillna(data[columns].mean())

In [82]:
data.dropna(inplace=True)

In [83]:
data.isna().sum()

﻿RID                                                            0
Test_data                                                       0
Age                                                             0
Gender                                                          0
Year_education                                                  0
                                                               ..
Volume (Cortical Parcellation) of RightMiddleTemporal           0
Surface Area of RightMiddleTemporal                             0
Cortical Thickness Average of RightMiddleTemporal               0
Cortical Thickness Standard Deviation of RightMiddleTemporal    0
Volume (WM Parcellation) of FourthVentricle                     0
Length: 361, dtype: int64

##### Cognitive dataframe

In [84]:
cognitive.isna().sum()

﻿RID                      0
Test_data                 0
CDRSB                     0
ADAS11                    5
ADAS13                   14
MMSE                      0
RAVLT_immediate           6
RAVLT_learning            6
RAVLT_forgetting          6
RAVLT_perc_forgetting    11
dtype: int64

Cognitive dataframe shape is (1737, 10), since missing values are less than 5% I will drop it

In [85]:
cognitive.dropna(inplace=True)

In [86]:
cognitive.isna().sum().sum()

np.int64(0)

##### Diagnosis dataframe

In [87]:
print(diagnosis.shape)
diagnosis.isna().sum()

(1737, 4)


﻿RID           0
Test_data      0
Diagnosis      0
FDG_PET      434
dtype: int64

In [88]:
diagnosis.dtypes

﻿RID         int64
Test_data    int64
Diagnosis      str
FDG_PET        str
dtype: object

In [89]:
# Even though NANs are big portion of FDG_PET after numerous attempts I decided to drop it

diagnosis.dropna()

,﻿RID,Test_data,Diagnosis,FDG_PET
0,2,0,CN,"1,36926"
1,3,0,AD,"1,09079"
3,5,0,CN,"1,29799"
6,8,0,CN,"1,27628"
7,10,0,AD,"1,11881"
...,...,...,...,...
1730,5275,0,AD,"1,03993"
1731,5282,0,SMC,"1,13549"
1732,5287,0,SMC,"1,58312"
1733,5295,0,SMC,"1,16317"


##### Merging dataframes

In [90]:
print(diagnosis.columns.tolist())

['\ufeffRID', 'Test_data', 'Diagnosis', 'FDG_PET']


In [91]:
print(cognitive.columns.tolist())

['\ufeffRID', 'Test_data', 'CDRSB', 'ADAS11', 'ADAS13', 'MMSE', 'RAVLT_immediate', 'RAVLT_learning', 'RAVLT_forgetting', 'RAVLT_perc_forgetting']


In [92]:
print(data.columns.tolist())

['\ufeffRID', 'Test_data', 'Age', 'Gender', 'Year_education', 'Ethnicity', 'Race', 'Marital_status', 'High_risk_ApoE4', 'Ventricles', 'Hippocampus', 'WholeBrain', 'Entorhinal', 'Fusiform', 'MidTemp', 'Intra cranial volume', 'Volume (WM Parcellation) of RightPallidum', 'Volume (Cortical Parcellation) of RightParacentral', 'Surface Area of RightParacentral', 'Cortical Thickness Average of RightParacentral', 'Cortical Thickness Standard Deviation of RightParacentral', 'Volume (Cortical Parcellation) of RightParahippocampal', 'Surface Area of RightParahippocampal', 'Cortical Thickness Average of RightParahippocampal', 'Cortical Thickness Standard Deviation of RightParahippocampal', 'Volume (Cortical Parcellation) of RightParsOpercularis', 'Surface Area of RightParsOpercularis', 'Cortical Thickness Average of RightParsOpercularis', 'Cortical Thickness Standard Deviation of RightParsOpercularis', 'Volume (Cortical Parcellation) of RightParsOrbitalis', 'Surface Area of RightParsOrbitalis', 'C

In [93]:
print(diagnosis["Test_data"].nunique(), len(diagnosis))
print(cognitive["Test_data"].nunique(), len(cognitive))
print(data["Test_data"].nunique(), len(data))

2 1737
2 1714
2 1171


In [94]:
print(diagnosis["Test_data"].is_unique)
print(cognitive["Test_data"].is_unique)
print(data["Test_data"].is_unique)

False
False
False


In [95]:
set(diagnosis.columns) & set(cognitive.columns) & set(data.columns)

{'Test_data', '\ufeffRID'}

In [96]:
print(diagnosis.columns[0])

﻿RID


In [97]:
diagnosis.columns = diagnosis.columns.str.replace("\ufeff", "")
cognitive.columns = cognitive.columns.str.replace("\ufeff", "")
data.columns = data.columns.str.replace("\ufeff", "")

In [98]:
sorted(set(diagnosis.columns)
       & set(cognitive.columns)
       & set(data.columns))

['RID', 'Test_data']

In [99]:
merged = pd.merge(diagnosis, cognitive, on="RID", how="inner")
df = pd.merge(merged, data, on="RID", how="inner")

df.head()

,RID,Test_data_x,Diagnosis,FDG_PET,Test_data_y,CDRSB,ADAS11,ADAS13,MMSE,RAVLT_immediate,...,Cortical Thickness Standard Deviation of RightLingual,Volume (Cortical Parcellation) of RightMedialOrbitofrontal,Surface Area of RightMedialOrbitofrontal,Cortical Thickness Average of RightMedialOrbitofrontal,Cortical Thickness Standard Deviation of RightMedialOrbitofrontal,Volume (Cortical Parcellation) of RightMiddleTemporal,Surface Area of RightMiddleTemporal,Cortical Thickness Average of RightMiddleTemporal,Cortical Thickness Standard Deviation of RightMiddleTemporal,Volume (WM Parcellation) of FourthVentricle
0,2,0,CN,"1,36926",0,0,"10,67","18,67",28,44.0,...,"0,694",3835,1622,"2,077","0,746",15683,4272,"3,028","0,649",4396
1,3,0,AD,"1,09079",0,"4,5",22,31,20,22.0,...,"0,591",3681,1734,"1,942","0,696",10387,3316,"2,545","0,686",3304
2,4,1,LMCI,NaN,1,1,"14,33","21,33",27,37.0,...,"0,588",4060,1728,"2,18","0,607",11156,3598,"2,67","0,631",1338
3,5,0,CN,"1,29799",0,0,"8,67","14,67",29,37.0,...,"0,628",5180,1868,"2,543","0,709",11579,3387,"2,911","0,66",1623
4,6,0,LMCI,NaN,0,"0,5","18,67","25,67",25,30.0,...,"0,631",3078,1241,"2,141","0,701",9641,2781,"2,9","0,727",1035


##### Standardising numerical columns

In [100]:
num_col = df.select_dtypes(include=["int64", "float64"]).columns
scaler = StandardScaler()
df[num_col] = scaler.fit_transform(df[num_col])

In [101]:
df[num_col].head()

,RID,Test_data_x,Test_data_y,MMSE,RAVLT_immediate,RAVLT_learning,RAVLT_forgetting,Test_data,Year_education,High_risk_ApoE4,Ventricles,Hippocampus,WholeBrain,Entorhinal,Fusiform,MidTemp,Intra cranial volume
0,-1.187937,-0.499190,-0.499190,0.296924,0.696566,-0.085493,0.675724,-0.499190,0.067955,-0.861262,3.445557,1.360042,1.931684,0.959678,-0.170965,2.926899,2.730554
1,-1.187405,-0.499190,-0.499190,-2.749263,-1.087998,-1.185504,-0.124776,-0.499190,0.752238,0.668403,1.955825,-1.224076,1.022353,-2.164304,-0.576140,-0.280845,2.343440
2,-1.186873,2.003244,2.003244,-0.083849,0.128750,1.014518,-0.124776,2.003244,-1.984891,-0.861262,-0.037069,0.103529,1.251256,0.705675,0.782137,0.121387,0.883516
3,-1.186342,-0.499190,-0.499190,0.677698,0.128750,-0.085493,-0.124776,-0.499190,0.067955,-0.861262,-0.282582,0.279972,0.902213,1.294858,2.995396,0.795371,0.649504
4,-1.185810,-0.499190,-0.499190,-0.845396,-0.439066,-1.185504,0.275474,-0.499190,-0.958468,-0.861262,-0.027280,-1.199237,-0.819064,-1.527986,0.369267,-0.489884,-0.288115


In [102]:
# For whatever reason there still were NANs in FDG_PET
df = df.dropna()

In [103]:
df.head(50)

,RID,Test_data_x,Diagnosis,FDG_PET,Test_data_y,CDRSB,ADAS11,ADAS13,MMSE,RAVLT_immediate,...,Cortical Thickness Standard Deviation of RightLingual,Volume (Cortical Parcellation) of RightMedialOrbitofrontal,Surface Area of RightMedialOrbitofrontal,Cortical Thickness Average of RightMedialOrbitofrontal,Cortical Thickness Standard Deviation of RightMedialOrbitofrontal,Volume (Cortical Parcellation) of RightMiddleTemporal,Surface Area of RightMiddleTemporal,Cortical Thickness Average of RightMiddleTemporal,Cortical Thickness Standard Deviation of RightMiddleTemporal,Volume (WM Parcellation) of FourthVentricle
0,-1.187937,-0.499190,CN,"1,36926",-0.499190,0,"10,67","18,67",0.296924,0.696566,...,"0,694",3835,1622,"2,077","0,746",15683,4272,"3,028","0,649",4396
1,-1.187405,-0.499190,AD,"1,09079",-0.499190,"4,5",22,31,-2.749263,-1.087998,...,"0,591",3681,1734,"1,942","0,696",10387,3316,"2,545","0,686",3304
3,-1.186342,-0.499190,CN,"1,29799",-0.499190,0,"8,67","14,67",0.677698,0.128750,...,"0,628",5180,1868,"2,543","0,709",11579,3387,"2,911","0,66",1623
6,-1.184746,-0.499190,CN,"1,27628",-0.499190,0,5,7,0.296924,1.264381,...,"0,541",3435,1412,"2,274","0,676",10469,3533,"2,601","0,655",2098
7,-1.183682,-0.499190,AD,"1,11881",-0.499190,5,"12,33","24,33",-1.226169,-1.250231,...,"0,541",3158,1381,"2,175","0,533",10209,3368,"2,622","0,611",1046
8,-1.181555,-0.499190,CN,"1,25699",-0.499190,0,"4,33","8,33",0.677698,0.777682,...,"0,613",3034,1295,"2,145","0,82",10493,3046,"2,85","0,714",1296
10,-1.180491,2.003244,CN,"1,39543",2.003244,0,"10,33","14,33",0.296924,0.372099,...,"0,509",4337,1660,"2,517","0,833",8911,2832,"2,735","0,617",1998
12,-1.177832,-0.499190,CN,"1,38279",-0.499190,0,"6,67","9,67",1.058471,1.426614,...,"0,543",4536,1748,"2,432","0,673",9203,2692,"3,025","0,596",1493
14,-1.176768,-0.499190,CN,"1,36422",-0.499190,0,4,8,-0.464622,0.372099,...,"0,475",4215,1771,"2,235","0,559",10936,3375,"2,736","0,697",1997
21,-1.166131,2.003244,CN,"1,30841",2.003244,0,7,10,0.677698,1.426614,...,"0,543",4621,1763,"2,413","0,761",10321,2966,"2,95","0,653",1474


In [106]:
selected_features = df.drop(columns=['RID','Test_data'])
# Target
y = selected_features["Diagnosis"]

# Features
X = selected_features.drop(columns="Diagnosis")

print("Selected Features:")
selected_features.head()

Selected Features:


,Test_data_x,Diagnosis,FDG_PET,Test_data_y,CDRSB,ADAS11,ADAS13,MMSE,RAVLT_immediate,RAVLT_learning,...,Cortical Thickness Standard Deviation of RightLingual,Volume (Cortical Parcellation) of RightMedialOrbitofrontal,Surface Area of RightMedialOrbitofrontal,Cortical Thickness Average of RightMedialOrbitofrontal,Cortical Thickness Standard Deviation of RightMedialOrbitofrontal,Volume (Cortical Parcellation) of RightMiddleTemporal,Surface Area of RightMiddleTemporal,Cortical Thickness Average of RightMiddleTemporal,Cortical Thickness Standard Deviation of RightMiddleTemporal,Volume (WM Parcellation) of FourthVentricle
0,-0.49919,CN,"1,36926",-0.49919,0,"10,67","18,67",0.296924,0.696566,-0.085493,...,"0,694",3835,1622,"2,077","0,746",15683,4272,"3,028","0,649",4396
1,-0.49919,AD,"1,09079",-0.49919,"4,5",22,31,-2.749263,-1.087998,-1.185504,...,"0,591",3681,1734,"1,942","0,696",10387,3316,"2,545","0,686",3304
3,-0.49919,CN,"1,29799",-0.49919,0,"8,67","14,67",0.677698,0.128750,-0.085493,...,"0,628",5180,1868,"2,543","0,709",11579,3387,"2,911","0,66",1623
6,-0.49919,CN,"1,27628",-0.49919,0,5,7,0.296924,1.264381,1.014518,...,"0,541",3435,1412,"2,274","0,676",10469,3533,"2,601","0,655",2098
7,-0.49919,AD,"1,11881",-0.49919,5,"12,33","24,33",-1.226169,-1.250231,-0.818834,...,"0,541",3158,1381,"2,175","0,533",10209,3368,"2,622","0,611",1046


In [108]:
categorical_cols = X.select_dtypes(include=["object", "string"]).columns

encoder = OneHotEncoder(
    sparse_output=False,
    handle_unknown="ignore"
)

encoded = encoder.fit_transform(X[categorical_cols])

encoded_df = pd.DataFrame(
    encoded,
    columns=encoder.get_feature_names_out(categorical_cols),
    index=X.index
)

X = pd.concat(
    [
        X.drop(columns=categorical_cols),
        encoded_df
    ],
    axis=1
)

X.head()

,Test_data_x,Test_data_y,MMSE,RAVLT_immediate,RAVLT_learning,RAVLT_forgetting,Year_education,High_risk_ApoE4,Ventricles,Hippocampus,...,Volume (WM Parcellation) of FourthVentricle_796,Volume (WM Parcellation) of FourthVentricle_845,Volume (WM Parcellation) of FourthVentricle_857,Volume (WM Parcellation) of FourthVentricle_889,Volume (WM Parcellation) of FourthVentricle_910,Volume (WM Parcellation) of FourthVentricle_911,Volume (WM Parcellation) of FourthVentricle_950,Volume (WM Parcellation) of FourthVentricle_962,Volume (WM Parcellation) of FourthVentricle_971,Volume (WM Parcellation) of FourthVentricle_985
0,-0.49919,-0.49919,0.296924,0.696566,-0.085493,0.675724,0.067955,-0.861262,3.445557,1.360042,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,-0.49919,-0.49919,-2.749263,-1.087998,-1.185504,-0.124776,0.752238,0.668403,1.955825,-1.224076,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,-0.49919,-0.49919,0.677698,0.128750,-0.085493,-0.124776,0.067955,-0.861262,-0.282582,0.279972,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,-0.49919,-0.49919,0.296924,1.264381,1.014518,-0.525026,0.752238,-0.861262,-0.960478,-0.572265,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,-0.49919,-0.49919,-1.226169,-1.250231,-0.818834,0.275474,-1.300609,0.668403,-0.603348,-1.081894,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [109]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [110]:
rf_classifier = RandomForestClassifier()
rf_classifier.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [112]:
train_accuracy = rf_classifier.score(X_train, y_train)
test_accuracy = rf_classifier.score(X_test, y_test)

print("Training Accuracy:", train_accuracy*100)
print("Testing Accuracy:", test_accuracy*100)

Training Accuracy: 100.0
Testing Accuracy: 50.602409638554214


## Conclusion

This project explored the use of machine learning for Alzheimer's disease classification by combining MRI volumetric measurements, cognitive assessment scores, and clinical diagnostic information into a single dataset. The preprocessing pipeline included data cleaning, handling missing values, merging multiple datasets, feature scaling, categorical encoding, and feature selection before training a classification model.

The trained model achieved **100% training accuracy** but only **50% testing accuracy**. This large performance gap indicates that the model has **overfit** the training data. Rather than learning general patterns associated with Alzheimer's disease, the model appears to have memorized characteristics of the training set, resulting in poor generalization to unseen patients.

Several factors may contribute to this behaviour:
- The dataset is relatively small after merging the available sources.
- The selected features may not provide enough predictive information.
- The model complexity may be too high for the amount of available data.
- Additional preprocessing or feature engineering may be required.

Future improvements could include:
- Performing cross-validation instead of relying on a single train-test split.
- Comparing multiple classification algorithms such as Logistic Regression, Support Vector Machines, Random Forest, Gradient Boosting, and XGBoost.
- Applying feature selection or dimensionality reduction techniques to remove redundant variables.
- Tuning model hyperparameters using GridSearchCV or RandomizedSearchCV.
- Investigating class balance and applying resampling techniques if necessary.
- Evaluating performance using precision, recall, F1-score, and ROC-AUC in addition to accuracy.

Overall, while the current model does not generalize well, the project demonstrates a complete end-to-end machine learning workflow, including data preprocessing, integration of multiple datasets, feature engineering, model training, and evaluation. The obtained results highlight the importance of validating models on unseen data and provide a solid foundation for further experimentation and model improvement.